In [1]:
# ============================================================
#  CSV → Parquet Converter  |  Colab-ready, single cell
#  Drop your CSV in Colab's file section, run this, download
# ============================================================

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import os, math
from google.colab import files

# ── CONFIG ──────────────────────────────────────────────────
CSV_FILE   = "master_1m_2026_04.csv"   # ← change if needed
OUT_FOLDER = "parquet_out"             # folder written in Colab
CHUNK_SIZE = 100_000                   # rows per chunk (safe for 97 MB)
COMPRESSION = "snappy"                 # fast read + good ratio
# ────────────────────────────────────────────────────────────

# 1. INSPECT ─────────────────────────────────────────────────
csv_mb = os.path.getsize(CSV_FILE) / 1e6
print(f"CSV size   : {csv_mb:.1f} MB")

peek = pd.read_csv(CSV_FILE, nrows=5)
total_cols = len(peek.columns)
print(f"Columns    : {total_cols}")
print(f"Column list: {peek.columns.tolist()}\n")

# Count total rows without loading full file
total_rows = sum(1 for _ in open(CSV_FILE)) - 1   # subtract header
print(f"Total rows : {total_rows:,}")
n_chunks   = math.ceil(total_rows / CHUNK_SIZE)
print(f"Chunks     : {n_chunks} × {CHUNK_SIZE:,} rows\n")

# 2. DTYPE MAP (auto-downcast after first chunk) ─────────────
def optimize_dtypes(df: pd.DataFrame) -> pd.DataFrame:
    for col in df.select_dtypes("object").columns:
        if df[col].nunique() / len(df) < 0.5:      # low cardinality → category
            df[col] = df[col].astype("category")
    for col in df.select_dtypes("int64").columns:
        df[col] = pd.to_numeric(df[col], downcast="integer")
    for col in df.select_dtypes("float64").columns:
        df[col] = pd.to_numeric(df[col], downcast="float")
    return df

# 3. CHUNK → PARQUET ─────────────────────────────────────────
os.makedirs(OUT_FOLDER, exist_ok=True)
writer      = None
schema      = None
rows_written = 0

reader = pd.read_csv(CSV_FILE, chunksize=CHUNK_SIZE, low_memory=False)

for i, chunk in enumerate(reader, 1):
    chunk = optimize_dtypes(chunk)

    table = pa.Table.from_pandas(chunk, preserve_index=False)

    if writer is None:
        schema = table.schema
        writer = pq.ParquetWriter(
            f"{OUT_FOLDER}/data.parquet",
            schema,
            compression=COMPRESSION
        )

    writer.write_table(table)
    rows_written += len(chunk)
    print(f"  chunk {i}/{n_chunks}  →  {rows_written:,} rows written", end="\r")

writer.close()
print(f"\nDone. {rows_written:,} rows written.\n")

# 4. VALIDATE ────────────────────────────────────────────────
pq_meta   = pq.read_metadata(f"{OUT_FOLDER}/data.parquet")
pq_rows   = pq_meta.num_rows
pq_mb     = os.path.getsize(f"{OUT_FOLDER}/data.parquet") / 1e6

print("── Validation ──────────────────────────────")
print(f"CSV rows    : {total_rows:,}")
print(f"Parquet rows: {pq_rows:,}")
print(f"Match       : {'✅ YES' if total_rows == pq_rows else '❌ MISMATCH – check your CSV!'}")
print(f"Parquet size: {pq_mb:.1f} MB  (was {csv_mb:.1f} MB)")
print(f"Compression : {(1 - pq_mb/csv_mb)*100:.0f}% smaller")
print("────────────────────────────────────────────\n")

# 5. QUICK LOAD TEST ─────────────────────────────────────────
sample = pq.read_table(f"{OUT_FOLDER}/data.parquet").slice(0, 5).to_pandas()
print("Sample (first 5 rows):")
print(sample.to_string(index=False))
print()

# 6. DOWNLOAD TO YOUR PC ─────────────────────────────────────
print("Downloading parquet file to your PC...")
files.download(f"{OUT_FOLDER}/data.parquet")
print("Download started. Save it somewhere safe.")

# ── HOW TO RELOAD LATER (backtesting) ───────────────────────
# 1. Drag data.parquet back into Colab files panel
# 2. Run:
#      import pyarrow.parquet as pq
#      df = pq.read_table("data.parquet").to_pandas()
#
# Load only specific columns (faster):
#      df = pq.read_table("data.parquet", columns=["datetime","close","volume"]).to_pandas()
#
# Load a date slice (if you add date partitioning later):
#      import pyarrow.dataset as ds
#      dataset = ds.dataset("data.parquet")
#      df = dataset.to_table(filter=ds.field("date") == "2026-04-01").to_pandas()

CSV size   : 101.8 MB
Columns    : 7
Column list: ['datetime', 'ticker', 'open', 'high', 'low', 'close', 'volume']

Total rows : 1,118,629
Chunks     : 12 × 100,000 rows

  chunk 12/12  →  1,118,629 rows written
Done. 1,118,629 rows written.

── Validation ──────────────────────────────
CSV rows    : 1,118,629
Parquet rows: 1,118,629
Match       : ✅ YES
Parquet size: 18.1 MB  (was 101.8 MB)
Compression : 82% smaller
────────────────────────────────────────────

Sample (first 5 rows):
           datetime    ticker       open       high        low      close  volume
2026-04-01 09:15:00 ABCAPITAL 302.350006 304.549988 301.549988 303.799988       0
2026-04-01 09:16:00 ABCAPITAL 303.799988 303.899994 303.049988 303.049988   44003
2026-04-01 09:17:00 ABCAPITAL 302.899994 303.549988 302.899994 303.549988   24951
2026-04-01 09:18:00 ABCAPITAL 303.299988 304.049988 303.299988 304.049988   23014
2026-04-01 09:19:00 ABCAPITAL 303.950012 304.100006 302.950012 302.950012   32610



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download started. Save it somewhere safe.


In [2]:
# ── STEP 1: Re-upload your parquet file ─────────────────────
from google.colab import files
import pyarrow.parquet as pq

print("Select your data.parquet file...")
uploaded = files.upload()   # click 'Choose Files' → pick data.parquet

# ── STEP 2: Load it ─────────────────────────────────────────
filename = list(uploaded.keys())[0]
print(f"\nLoaded: {filename}")

df = pq.read_table(filename).to_pandas()

print(f"Rows   : {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
print(df.head())

Select your data.parquet file...


Saving 1m_2026_04.parquet to 1m_2026_04.parquet

Loaded: 1m_2026_04.parquet
Rows   : 1,118,629
Columns: 7
              datetime     ticker        open        high         low  \
0  2026-04-01 09:15:00  ABCAPITAL  302.350006  304.549988  301.549988   
1  2026-04-01 09:16:00  ABCAPITAL  303.799988  303.899994  303.049988   
2  2026-04-01 09:17:00  ABCAPITAL  302.899994  303.549988  302.899994   
3  2026-04-01 09:18:00  ABCAPITAL  303.299988  304.049988  303.299988   
4  2026-04-01 09:19:00  ABCAPITAL  303.950012  304.100006  302.950012   

        close  volume  
0  303.799988       0  
1  303.049988   44003  
2  303.549988   24951  
3  304.049988   23014  
4  302.950012   32610  


In [3]:
import pandas as pd
import pyarrow.parquet as pq

df = pq.read_table("1m_2026_04.parquet").to_pandas()

print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nDtypes:\n{df.dtypes}")
print(f"\nSample:\n{df.head(3).to_string()}")

Shape: 1,118,629 rows × 7 columns

Columns: ['datetime', 'ticker', 'open', 'high', 'low', 'close', 'volume']

Dtypes:
datetime    category
ticker      category
open         float32
high         float32
low          float32
close        float32
volume         int32
dtype: object

Sample:
              datetime     ticker        open        high         low       close  volume
0  2026-04-01 09:15:00  ABCAPITAL  302.350006  304.549988  301.549988  303.799988       0
1  2026-04-01 09:16:00  ABCAPITAL  303.799988  303.899994  303.049988  303.049988   44003
2  2026-04-01 09:17:00  ABCAPITAL  302.899994  303.549988  302.899994  303.549988   24951


In [4]:
import pandas as pd
import pyarrow.parquet as pq

df = pq.read_table("1m_2026_04.parquet").to_pandas()
df["datetime"] = pd.to_datetime(df["datetime"])

summary = (
    df.groupby("ticker", observed=True)
    .agg(
        first_date=("datetime", "min"),
        last_date=("datetime", "max"),
        total_candles=("datetime", "count"),
        trading_days=("datetime", lambda x: x.dt.date.nunique())
    )
    .reset_index()
    .sort_values("total_candles", ascending=False)
)

print(f"Total stocks : {summary.shape[0]}")
print(f"Overall range: {df['datetime'].min()} → {df['datetime'].max()}")
print(f"\n{summary.to_string(index=False)}")

Total stocks : 156
Overall range: 2026-04-01 09:15:00 → 2026-04-30 15:29:00

    ticker          first_date           last_date  total_candles  trading_days
 ABCAPITAL 2026-04-01 09:15:00 2026-04-30 15:29:00           7199            20
  ADANIENT 2026-04-01 09:15:00 2026-04-30 15:29:00           7199            20
ADANIGREEN 2026-04-01 09:15:00 2026-04-30 15:29:00           7199            20
ADANIPORTS 2026-04-01 09:15:00 2026-04-30 15:29:00           7199            20
  AXISBANK 2026-04-01 09:15:00 2026-04-30 15:29:00           7199            20
 AMBUJACEM 2026-04-01 09:15:00 2026-04-30 15:29:00           7199            20
APOLLOHOSP 2026-04-01 09:15:00 2026-04-30 15:29:00           7199            20
ASIANPAINT 2026-04-01 09:15:00 2026-04-30 15:29:00           7199            20
AUROPHARMA 2026-04-01 09:15:00 2026-04-30 15:29:00           7199            20
BAJFINANCE 2026-04-01 09:15:00 2026-04-30 15:29:00           7199            20
BAJAJFINSV 2026-04-01 09:15:00 2026-04-30 1